<a href="https://colab.research.google.com/github/adrianserrano55/FPL_Predictive_Model/blob/main/fplPredictions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [50]:
import pandas as pd
import numpy as np
import requests
import os
from sklearn.ensemble import RandomForestRegressor
import warnings

warnings.filterwarnings('ignore')

class PlayerProfile:
    """Organizes player data, incorporating position-specific metrics and FDR."""
    def __init__(self, player_data):
        self.player_id = player_data['player_id']
        self.player_name = player_data['player_name']
        self.position = player_data['position']
        self.team_id = player_data['team_id']
        self.team_name = player_data['team_name']
        self.status = player_data['status']
        self.form = player_data['form']
        self.ep_next = player_data['ep_next']
        self.minutes = player_data['minutes']
        self.total_points = player_data['total_points']
        self.fdr = player_data.get('fdr', 3.0)

        # Retained metrics
        self.xG = player_data.get('xG', 0.0)
        self.xA = player_data.get('xA', 0.0)
        self.xGC = player_data.get('xGC', 0.0)

        # Advanced metrics
        self.IGP = player_data.get('IGP', 0.0)
        self.ICSP = player_data.get('ICSP', 0.0)
        self.xT = player_data.get('xT', 0.0)

    def get_feature_vector(self, h2h_val):
        """Returns position-tailored feature vector including FDR and H2H."""
        if self.position in ['GK', 'DEF']:
            return [self.form, self.fdr, self.xGC, self.ICSP, h2h_val]
        else:
            return [self.form, self.fdr, self.xG, self.xA, self.IGP, self.xT, h2h_val]

    def get_feature_columns(self):
        """Returns column names matching the positional feature vector."""
        if self.position in ['GK', 'DEF']:
            return ['form', 'fdr', 'xGC', 'ICSP', 'H2H_Score']
        else:
            return ['form', 'fdr', 'xG', 'xA', 'IGP', 'xT', 'H2H_Score']


class FPLProfileRepositoryModel:
    def __init__(self, history_file='fpl_profile_repository.csv'):
        self.history_file = history_file
        self.position_map = {1: 'GK', 2: 'DEF', 3: 'MID', 4: 'FWD'}
        print("Connecting to FPL API and initializing updated repository with FDR...")
        self.current_season = "2026/27"
        self.player_profiles, self.next_fixtures_map, self.current_gw = self.fetch_current_fpl_data()
        self.models = {}

    def fetch_current_fpl_data(self):
        """Pulls live player pools for players with > 1 total point, fixture difficulties (FDR), and metrics."""
        url_static = "https://fantasy.premierleague.com/api/bootstrap-static/"
        url_fixtures = "https://fantasy.premierleague.com/api/fixtures/"

        teams_dict = {}
        fixtures_map = {}
        current_gw = 4

        try:
            res_static = requests.get(url_static, timeout=15)
            if res_static.status_code == 200:
                data = res_static.json()
                for team in data.get('teams', []):
                    teams_dict[team['id']] = team['name']

            res_fix = requests.get(url_fixtures, timeout=15)
            if res_fix.status_code == 200:
                fix_data = res_fix.json()
                current_gw = next((f['event'] for f in fix_data if f.get('event') is not None and not f.get('finished', True)), 4)

                for f in fix_data:
                    if f.get('event') == current_gw:
                        h_id = f['team_h']
                        a_id = f['team_a']
                        h_name = teams_dict.get(h_id, f"Team {h_id}")
                        a_name = teams_dict.get(a_id, f"Team {a_id}")

                        fixtures_map[h_id] = {'opponent_name': a_name, 'fixture_label': f"vs {a_name} (H)", 'fdr': float(f.get('team_h_difficulty', 3))}
                        fixtures_map[a_id] = {'opponent_name': h_name, 'fixture_label': f"at {h_name} (A)", 'fdr': float(f.get('team_a_difficulty', 3))}

            profiles = {}
            if res_static.status_code == 200:
                for player in data.get('elements', []):
                    pos_id = player['element_type']
                    t_id = player['team']
                    team_name = teams_dict.get(t_id, f"Team {t_id}")
                    total_pts = int(player.get('total_points', 0))

                    # Create profile only for players with more than 1 total point
                    if pos_id in self.position_map and total_pts > 1:
                        team_fdr = fixtures_map.get(t_id, {}).get('fdr', 3.0)
                        raw_data = {
                            'player_id': player['id'],
                            'player_name': f"{player['first_name']} {player['second_name']}",
                            'position': self.position_map[pos_id],
                            'team_id': t_id,
                            'team_name': team_name,
                            'status': player.get('status', 'a'),
                            'ep_next': float(player.get('ep_next', 0.0) or 0.0),
                            'form': float(player.get('form', 0.0)),
                            'minutes': player.get('minutes', 0),
                            'total_points': total_pts,
                            'fdr': team_fdr,
                            'xG': float(player.get('expected_goals', 0.0) or 0.0),
                            'xA': float(player.get('expected_assists', 0.0) or 0.0),
                            'xGC': float(player.get('expected_goals_conceded', 0.0) or 0.0),
                            'IGP': float(player.get('influence', 0.0) or 0.0) / 100.0,
                            'ICSP': float(player.get('creativity', 0.0) or 0.0) / 100.0,
                            'xT': float(player.get('threat', 0.0) or 0.0) / 100.0
                        }
                        profiles[player['id']] = PlayerProfile(raw_data)
        except Exception as e:
            print(f"API Warning: {e}. Utilizing fallback structure.")
            profiles = {}

        if not profiles:
            for i in range(1, 500):
                pos = self.position_map[((i - 1) % 4) + 1]
                t_id = (i % 20) + 1
                raw_data = {
                    'player_id': i,
                    'player_name': f"Player {i} ({pos})",
                    'position': pos,
                    'team_id': t_id,
                    'team_name': f"Club {t_id}",
                    'status': 'a',
                    'ep_next': round(np.random.uniform(2.0, 12.0), 1),
                    'form': round(np.random.uniform(1.0, 10.0), 1),
                    'minutes': 900,
                    'total_points': np.random.randint(2, 15),
                    'fdr': float(np.random.randint(1, 6)),
                    'xG': round(np.random.uniform(0.1, 5.0), 2),
                    'xA': round(np.random.uniform(0.1, 4.0), 2),
                    'xGC': round(np.random.uniform(1.0, 8.0), 2),
                    'IGP': round(np.random.uniform(0.5, 3.0), 2),
                    'ICSP': round(np.random.uniform(0.1, 2.0), 2),
                    'xT': round(np.random.uniform(0.2, 4.0), 2)
                }
                profiles[i] = PlayerProfile(raw_data)
                fixtures_map[t_id] = {'opponent_name': 'Opponent FC', 'fixture_label': 'vs Opponent FC (H)', 'fdr': 3.0}

        return profiles, fixtures_map, current_gw

    def manage_repository(self, active_profiles):
        """Appends profile metrics, FDR, and real weekly match results into repository storage."""
        snapshot_records = []
        for p_id, profile in active_profiles.items():
            opp_info = self.next_fixtures_map.get(profile.team_id, {'opponent_name': 'Unknown'})
            snapshot_records.append({
                'season': self.current_season,
                'gameweek': self.current_gw,
                'player_id': profile.player_id,
                'player_name': profile.player_name,
                'team_name': profile.team_name,
                'opponent_name': opp_info['opponent_name'],
                'position': profile.position,
                'form': profile.form,
                'fdr': profile.fdr,
                'xG': profile.xG,
                'xA': profile.xA,
                'xGC': profile.xGC,
                'IGP': profile.IGP,
                'ICSP': profile.ICSP,
                'xT': profile.xT,
                'actual_points': profile.total_points
            })

        current_snapshot_df = pd.DataFrame(snapshot_records)

        if os.path.exists(self.history_file):
            history_df = pd.read_csv(self.history_file)
            already_logged = ((history_df['season'] == self.current_season) & (history_df['gameweek'] == self.current_gw)).any()
            if not already_logged:
                updated_history = pd.concat([history_df, current_snapshot_df], ignore_index=True)
                updated_history.to_csv(self.history_file, index=False)
                print(f"📁 Logged real GW {self.current_gw} data for season {self.current_season} into profile repository.")
            else:
                print(f"📁 GW {self.current_gw} data for {self.current_season} already exists in repository.")
        else:
            print(f"📁 Initializing empty profile-based repository: '{self.history_file}'...")
            current_snapshot_df.to_csv(self.history_file, index=False)

        return pd.read_csv(self.history_file)

    def calculate_strict_h2h_metric(self, history_df, player_id, opponent_name):
        """Strictly searches prior historical fixtures against this specific opponent. Returns NaN if zero prior meetings exist."""
        if history_df.empty:
            return np.nan

        h2h_matches = history_df[
            (history_df['player_id'] == player_id) &
            (history_df['opponent_name'] == opponent_name) &
            ~((history_df['season'] == self.current_season) & (history_df['gameweek'] == self.current_gw))
        ]

        if len(h2h_matches) > 0:
            return h2h_matches['actual_points'].mean()

        return np.nan

    def train_and_predict(self):
        active_profiles = {p_id: p for p_id, p in self.player_profiles.items() if p.total_points > 1}

        history_df = self.manage_repository(active_profiles)

        for position in ['GK', 'DEF', 'MID', 'FWD']:
            pos_train = history_df[history_df['position'] == position].copy()
            if len(pos_train) > 5:
                pos_train['H2H_Score'] = [
                    self.calculate_strict_h2h_metric(history_df, r['player_id'], r['opponent_name'])
                    for _, r in pos_train.iterrows()
                ]
                pos_train['H2H_Score'] = pos_train['H2H_Score'].fillna(4.5)

                if position in ['GK', 'DEF']:
                    features_cols = ['form', 'fdr', 'xGC', 'ICSP', 'H2H_Score']
                else:
                    features_cols = ['form', 'fdr', 'xG', 'xA', 'IGP', 'xT', 'H2H_Score']

                X = pos_train[features_cols]
                y = pos_train['actual_points']

                model = RandomForestRegressor(n_estimators=150, random_state=42, min_samples_leaf=2)
                model.fit(X, y)
                self.models[position] = model
            else:
                self.models[position] = None

        results_data = []
        for p_id, profile in active_profiles.items():
            fix_info = self.next_fixtures_map.get(profile.team_id, {'opponent_name': 'Unknown', 'fixture_label': 'Unknown', 'fdr': 3.0})
            h2h_score = self.calculate_strict_h2h_metric(history_df, profile.player_id, fix_info['opponent_name'])
            h2h_val_for_model = h2h_score if not np.isnan(h2h_score) else 4.5

            model = self.models.get(profile.position)
            if model:
                feat_vector = pd.DataFrame([profile.get_feature_vector(h2h_val_for_model)], columns=profile.get_feature_columns())
                pred = model.predict(feat_vector)[0]
            else:
                pred = profile.ep_next if profile.ep_next > 0.0 else profile.form * 0.85

            model_epn = np.clip(pred, 0.5, 28.0).round(2)

            results_data.append({
                'player_id': profile.player_id,
                'player_name': profile.player_name,
                'position': profile.position,
                'team_name': profile.team_name,
                'Fixture': fix_info['fixture_label'],
                'form': profile.form,
                'FDR': profile.fdr,
                'H2H_Score': h2h_score,
                'Model_EPN': model_epn,
                'primary_stat': profile.xG if profile.position in ['MID', 'FWD'] else profile.xGC
            })

        results_df = pd.DataFrame(results_data)

        print("\n" + "=" * 125)
        print(f"⚽ PROFILE-BASED MODEL WITH FDR ({self.current_season} - GW {self.current_gw})")
        print("=" * 125)

        for position in ['GK', 'DEF', 'MID', 'FWD']:
            pos_subset = results_df[results_df['position'] == position]
            top_30 = pos_subset.sort_values(by='Model_EPN', ascending=False).head(30)

            stat_label = "xG" if position in ['MID', 'FWD'] else "xGC"
            print(f"\n🏆 TOP 30 {position}S (Repository Records: {len(history_df)})")
            print("-" * 125)
            print(f"{'Player Name':<20} | {'Team':<12} | {'Fixture':<18} | {'Form':<5} | {'H2H':<5} | {'FDR':<5} | {stat_label:<5} | {'Model EPN':<10}")
            print("-" * 125)

            for _, row in top_30.iterrows():
                h2h_display = f"{row['H2H_Score']:.1f}" if not np.isnan(row['H2H_Score']) else "-"
                print(
                    f"{row['player_name'][:20]:<20} | "
                    f"{row['team_name'][:12]:<12} | "
                    f"{row['Fixture'][:18]:<18} | "
                    f"{row['form']:>5.1f} | "
                    f"{h2h_display:>5} | "
                    f"{row['FDR']:>5.1f} | "
                    f"{row['primary_stat']:>5.2f} | "
                    f"{row['Model_EPN']:>10.2f}"
                )
            print("-" * 125)

if __name__ == "__main__":
    model = FPLProfileRepositoryModel()
    model.train_and_predict()

Connecting to FPL API and initializing updated repository with FDR...
📁 Initializing empty profile-based repository: 'fpl_profile_repository.csv'...

⚽ PROFILE-BASED MODEL WITH FDR (2026/27 - GW 3)

🏆 TOP 30 GKS (Repository Records: 264)
-----------------------------------------------------------------------------------------------------------------------------
Player Name          | Team         | Fixture            | Form  | H2H   | FDR   | xGC   | Model EPN 
-----------------------------------------------------------------------------------------------------------------------------
James Trafford       | Leeds        | at Brighton (A)    |   6.0 |     - |   3.0 |  2.33 |      13.36
Konstantinos Tzolaki | Hull City    | vs Aston Villa (H) |  10.0 |     - |   3.0 |  3.12 |      13.35
David Raya Martín    | Arsenal      | vs Chelsea (H)     |   6.0 |     - |   4.0 |  0.53 |      13.03
Jordan Pickford      | Everton      | vs Man Utd (H)     |   5.0 |     - |   4.0 |  4.12 |      10.38
